# سبق 09 - میٹا کگنیشن ڈیزائن پیٹرن


## ترتیب

یہ نوٹ بک Microsoft Agent Framework کا استعمال کرتے ہوئے Metacognition ڈیزائن پیٹرن کی نمائش کرتی ہے۔

**ضروریات:**
- Azure OpenAI تنصیب ماحولیاتی متغیروں کے ذریعے ترتیب دی گئی ہو
- Azure CLI مستند شدہ (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## میٹاگنیشن کیا ہے؟

میٹاگنیشن کا مطلب ہے **سوچ کے بارے میں سوچنا**۔ AI ایجینٹس کے تناظر میں، اس کا مطلب ایسے ایجنٹس بنانا ہے جو:

- اپنے نتائج اور استدلالی عمل پر **خود عکاسی** کریں
- خامیوں کا **اندازہ لگائیں** اور بغیر کسی خاموش ناکامی کے بحالی کریں
- یہ **جائزہ لیں** کہ ان کے جوابات مکمل اور مددگار ہیں یا نہیں
- اپنی حکمت عملی کو **موزوں بنائیں** جب ابتدائی طریقہ کار ناکام ہو جائے (مثلاً، ایک بیک اپ نظام کی طرف واپس جانا)

ایک میٹاگنیشن ایجنٹ محض سوالات کا جواب نہیں دیتا — یہ اپنی کارکردگی کی نگرانی کرتا ہے اور فوری طور پر ایڈجسٹ کرتا ہے۔


## بنیادی اور بیک اپ آلات

ایک عام میٹاگنیشن پیٹرن **بیک اپ حکمت عملی** ہے۔ ایجنٹ پہلے بنیادی آلہ آزما تا ہے؛ اگر وہ ناکام ہو جائے (مثلاً، 404 ایرر)، ایجنٹ ناکامی کو پہچانتا ہے اور شفاف طریقے سے بیک اپ آلے پر سوئچ کر جاتا ہے۔

یہ حقیقی دنیا کے نظاموں کی عکاسی کرتا ہے جہاں بنیادی خدمات دستیاب نہیں ہو سکتیں اور ایجنٹ کو مسئلہ خود تشخیص کرنا ہوتا ہے اس سے پہلے کہ وہ متبادل راستہ منتخب کرے۔

نیچے ہم دو فلائٹ لوک اپ آلات کی وضاحت کرتے ہیں:
- **بنیادی** — پیرس، ٹوکیو، اور بارسلونا کا احاطہ کرتا ہے
- **بیک اپ** — برلن، سڈنی، اور نیو یارک سٹی کا احاطہ کرتا ہے


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## غلطی کی بازیابی کے ساتھ خود عکاسی کرتا ایجنٹ

نیچے دیے گئے ایجنٹ کو ہدایت دی گئی ہے کہ پہلے پرائمری فلائٹ سسٹم کو آزماے، خرابیوں کو پہچانے، اور شفاف طریقے سے بیک اپ سسٹم پر واپس چلا جائے۔ ہر جواب کے بعد وہ مختصراً خود کا جائزہ لیتا ہے کہ آیا اس نے صارف کے سوال کا مکمل جواب دیا ہے یا نہیں۔


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## خود تشخیصی کا نمونہ

میٹا کگنیشن کا ایک اور پہلو **خود تشخیص** ہے: ایک الگ ایجنٹ (یا ایک ہی ایجنٹ دوسری بار میں) جواب کو مکمل ہونے، درستگی، اور مددگار ہونے کے لئے جائزہ لیتا ہے۔

نیچے ہم ایک `ResponseEvaluator` ایجنٹ تخلیق کرتے ہیں جو ٹریول ایجنٹ کے جوابات کو تین جہتوں پر اسکور کرتا ہے۔


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## خلاصہ

اس سبق میں آپ نے سیکھا کہ مائیکروسافٹ ایجنٹ فریم ورک کا استعمال کرتے ہوئے **میتاکاگنیٹو ایجنٹس** کیسے بنائے جاتے ہیں:

- **خود عکاسی**: ایسے ایجنٹس جو اپنی سوچ کو مانیٹر کرتے ہیں اور شفاف انداز میں بتاتے ہیں کہ کیا ہوا۔
- **فال بیکس کے ساتھ خرابی کی بحالی**: ایک پرائمری + بیک اپ ٹول پیٹرن جہاں ایجنٹ ناکامیاں (مثلاً، 404 کی غلطیاں) detect کرتا ہے اور خودکار طریقے سے متبادل ذریعہ آزما لیتا ہے۔
- **خود تشخیص**: ایک الگ ایوالویٹر ایجنٹ جو جوابات کو مکمل پن، درستگی، اور مددگار ہونے کی بنیاد پر سکور دیتا ہے۔

یہ پیٹرنز ایجنٹس کو زیادہ مضبوط، شفاف، اور قابل اعتماد بناتے ہیں — جو کہ پروڈکشن فیلڈ میں تعیناتی کے لیے اہم خصوصیات ہیں۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
